In [1]:
import pandas as pd
import torch
df = pd.read_json("dataset.txt", lines=True)
df = df[["reviewText", "overall"]]
df.head(10), df['overall'].unique(), df.shape

(                                          reviewText  overall
 0  They look good and stick good! I just don't li...        4
 1  These stickers work like the review says they ...        5
 2  These are awesome and make my phone look so st...        5
 3  Item arrived in great time and was in perfect ...        4
 4  awesome! stays on, and looks great. can be use...        5
 5  These make using the home button easy. My daug...        3
 6  Came just as described.. It doesn't come unstu...        5
 7  it worked for the first week then it only char...        1
 8  Good case, solid build. Protects phone all aro...        5
 9  This is a fantastic case. Very stylish and pro...        5,
 array([4, 5, 3, 1, 2]),
 (11000, 2))

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
from torch.utils.data import dataloader
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['reviewText'].tolist(), df['overall'].tolist(), stratify = df['overall'], test_size=1/11, random_state = 42)
len(X_train), len(X_test), len(y_train), len(y_test)


(10000, 1000, 10000, 1000)

In [4]:
from transformers import BertTokenizer, BertForSequenceClassification 
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(X_train, padding='max_length', truncation=True, return_tensors="pt")
test_encodings = tokenizer(X_test, padding='max_length', truncation=True, return_tensors="pt")

y_train = [int(y) - 1 for y in y_train]
y_test = [int(y) - 1 for y in y_test]


c:\Users\sudwivedi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
class ReviewDataset(Dataset):
    def __init__(self, labels, encodings):
        self.labels = labels
        self.encodings = encodings
    
    def __getitem__(self, idx):
        item = { key: val[idx] for key, val in self.encodings.items() }
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    

train_dataset = ReviewDataset(y_train, train_encodings)
test_dataset = ReviewDataset(y_test, test_encodings)


In [6]:
train_loader = DataLoader(train_dataset, shuffle = True, batch_size = 32)
test_loader = DataLoader(test_dataset, batch_size=32)


In [27]:
batch = next(iter(train_loader))
batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape          

(torch.Size([32, 512]), torch.Size([32, 512]), torch.Size([32]))

In [7]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5).to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4888.78it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [8]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [9]:
from tqdm import tqdm
model.eval()
all_labels = []
all_predictions = []
for batch in tqdm(test_loader):
    input_ids = batch['input_ids']
    attention_masks = batch['attention_mask']
    labels = batch['labels']
    input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_masks)
    
    predictions = torch.argmax(outputs.logits, dim=1)
    all_labels.extend(labels.cpu().numpy())
    all_predictions.extend(predictions.cpu().numpy())

100%|██████████| 32/32 [00:19<00:00,  1.63it/s]


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Accuracy: {accuracy}")

print(classification_report(all_labels, all_predictions))
print(confusion_matrix(all_labels, all_predictions))

Accuracy: 0.073
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        96
           1       0.07      0.99      0.13        70
           2       0.00      0.00      0.00       111
           3       0.06      0.01      0.02       205
           4       1.00      0.00      0.01       518

    accuracy                           0.07      1000
   macro avg       0.23      0.20      0.03      1000
weighted avg       0.53      0.07      0.02      1000

[[  0  86   0  10   0]
 [  0  69   0   1   0]
 [  0 107   0   4   0]
 [  0 203   0   2   0]
 [  0 498   0  18   2]]


c:\Users\sudwivedi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sudwivedi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sudwivedi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [11]:
from torch.optim import AdamW
adam_optim = AdamW(model.parameters(), lr=5e-5)

In [12]:
model.train()

for epoch in range(5):
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids']
        attention_masks = batch['attention_mask']
        labels = batch['labels']
        input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        loss.backward()
        adam_optim.step()

 27%|██▋       | 83/313 [36:55<1:42:20, 26.70s/it]


KeyboardInterrupt: 

In [ ]:
from tqdm import tqdm
model.eval()
all_labels = []
all_predictions = []
for batch in tqdm(test_loader):
    input_ids = batch['input_ids']
    attention_masks = batch['attention_mask']
    labels = batch['labels']
    input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_masks)
    
    predictions = torch.argmax(outputs.logits, dim=1)
    all_labels.extend(labels.cpu().numpy())
    all_predictions.extend(predictions.cpu().numpy())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Accuracy: {accuracy}")

print(classification_report(all_labels, all_predictions))
print(confusion_matrix(all_labels, all_predictions))